# z609 - Walk-forward + Optuna (Etapa 10)
Mismos features (FE609). Unico cambio: validacion de un solo corte -> 3 cortes temporales (walk-forward), para elegir hiperparametros mas robustos.

In [1]:
!pip install -q lightgbm pyarrow optuna

In [2]:
import os
import numpy as np
import polars as pl
import lightgbm as lgb
import optuna
import warnings
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [3]:
PARAM = {
    'experimento': 'LGB07_WF',
    'kaggle_competition': 'labo-iii-2026-ba',
    'base_path': '/home/ds/exp/FE609/',
    'archivo_features': 'tb_features_FE609.parquet',
    'apredecir_path': '/home/ds/datasets/product_id_apredecir201912.txt',
    'horizonte_meses': 2,
    'periodo_ultimo_dato': 201912,
    'periodo_target_final': 202002,
    'semilla': 102103,
    'n_trials': 50
}

ruta = os.path.join('/home/ds/exp', PARAM['experimento'])
os.makedirs(ruta, exist_ok=True)
print(ruta)

/home/ds/exp/LGB07_WF


## 1. Cargar features y armar target

In [4]:
def periodo_a_meses(periodo: int) -> int:
    return (periodo // 100) * 12 + (periodo % 100)

df = pl.read_parquet(os.path.join(PARAM['base_path'], PARAM['archivo_features']))
df = df.sort(["product_id", "periodo"])

H = PARAM['horizonte_meses']

df = df.with_columns(
    pl.col("tn").shift(-H).over("product_id").alias("tn_target")
)
df = df.with_columns(
    (pl.col("periodo_m") + H).alias("periodo_target_m")
)

df_valido = df.filter(pl.col("tn_target").is_not_null())

## 2. Definir los 3 cortes (walk-forward)
Cada corte entrena con todo lo anterior y valida en un bloque de 2 meses, deslizando hacia adelante. El ultimo corte coincide con el split original (train &le;201910, valid 201911-201912).

In [5]:
cortes_calendario = [
    (201906, 201907, 201908),
    (201908, 201909, 201910),
    (201910, 201911, 201912),
]

folds = []
for train_max, valid_min, valid_max in cortes_calendario:
    m_train_max = periodo_a_meses(train_max)
    m_valid_min = periodo_a_meses(valid_min)
    m_valid_max = periodo_a_meses(valid_max)

    train_f = df_valido.filter(pl.col("periodo_target_m") <= m_train_max)
    valid_f = df_valido.filter(
        (pl.col("periodo_target_m") >= m_valid_min) & (pl.col("periodo_target_m") <= m_valid_max)
    )
    folds.append((train_f, valid_f))
    print(f"corte train<={train_max} valid={valid_min}-{valid_max}: train={train_f.height} valid={valid_f.height}")

corte train<=201906 valid=201907-201908: train=23645 valid=1788
corte train<=201908 valid=201909-201910: train=25433 valid=1816
corte train<=201910 valid=201911-201912: train=27249 valid=1827


## 3. Preparar matrices por fold

In [6]:
cols_excluir = {"tn", "tn_target", "tn_shift1", "periodo", "periodo_target_m", "nacimiento_m"}
features = [c for c in df.columns if c not in cols_excluir]
categoricas = [c for c in ["product_id", "cat1", "cat2", "cat3", "brand", "descripcion"] if c in features]

def a_pandas(tabla):
    pdf = tabla.select(features + ["tn_target"]).to_pandas()
    for c in categoricas:
        pdf[c] = pdf[c].astype("category")
    return pdf

datasets_por_fold = []
for train_f, valid_f in folds:
    train_pd = a_pandas(train_f)
    valid_pd = a_pandas(valid_f)

    X_train = train_pd[features]
    y_train = np.log1p(train_pd["tn_target"].clip(lower=0))
    X_valid = valid_pd[features]
    y_valid = np.log1p(valid_pd["tn_target"].clip(lower=0))

    dtrain = lgb.Dataset(X_train, label=y_train, categorical_feature=categoricas,
                          params={'feature_pre_filter': False})
    dvalid = lgb.Dataset(X_valid, label=y_valid, categorical_feature=categoricas, reference=dtrain,
                          params={'feature_pre_filter': False})
    datasets_por_fold.append((dtrain, dvalid))

## 4. Optuna sobre el promedio de los 3 folds

In [7]:
def objective(trial):
    params = {
        'objective': 'regression',
        'metric': 'rmse',
        'verbosity': -1,
        'seed': PARAM['semilla'],
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 15, 255),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 5, 200),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.5, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.5, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 7),
        'lambda_l1': trial.suggest_float('lambda_l1', 1e-8, 10.0, log=True),
        'lambda_l2': trial.suggest_float('lambda_l2', 1e-8, 10.0, log=True),
        'max_depth': trial.suggest_int('max_depth', -1, 15),
    }

    scores = []
    for dtrain, dvalid in datasets_por_fold:
        modelo = lgb.train(
            params,
            dtrain,
            num_boost_round=2000,
            valid_sets=[dvalid],
            callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
        )
        scores.append(modelo.best_score['valid_0']['rmse'])

    return float(np.mean(scores))

study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=PARAM['semilla']))
study.optimize(objective, n_trials=PARAM['n_trials'], show_progress_bar=True)

print("mejor rmse promedio (3 folds):", study.best_value)
print("mejores params:", study.best_params)

  0%|          | 0/50 [00:00<?, ?it/s]

mejor rmse promedio (3 folds): 0.44718140721682337
mejores params: {'learning_rate': 0.011997528564839596, 'num_leaves': 110, 'min_data_in_leaf': 56, 'feature_fraction': 0.8380549780622345, 'bagging_fraction': 0.7380770088618046, 'bagging_freq': 3, 'lambda_l1': 4.358281105200902e-05, 'lambda_l2': 3.7666480784105863e-06, 'max_depth': 0}


## 5. Reentrenar final con el ultimo corte (train&le;201910, valid 201911-201912)
Mismo esquema que las etapas anteriores, para mantener comparabilidad del submit final.

In [8]:
mejores_params = dict(study.best_params)
mejores_params.update({
    'objective': 'regression',
    'metric': 'rmse',
    'verbosity': -1,
    'seed': PARAM['semilla']
})

dtrain_final, dvalid_final = datasets_por_fold[-1]

modelo_final = lgb.train(
    mejores_params,
    dtrain_final,
    num_boost_round=2000,
    valid_sets=[dtrain_final, dvalid_final],
    valid_names=['train', 'valid'],
    callbacks=[lgb.early_stopping(stopping_rounds=100), lgb.log_evaluation(period=100)]
)

print("mejor iteracion:", modelo_final.best_iteration)

Training until validation scores don't improve for 100 rounds
[100]	train's rmse: 0.632258	valid's rmse: 0.673106
[200]	train's rmse: 0.42226	valid's rmse: 0.503919
[300]	train's rmse: 0.366996	valid's rmse: 0.482686
[400]	train's rmse: 0.338341	valid's rmse: 0.481421
Early stopping, best iteration is:
[353]	train's rmse: 0.350894	valid's rmse: 0.480887
mejor iteracion: 353


## 6. Prediccion para 202002 y submit

In [9]:
futuro = df.filter(pl.col("periodo") == PARAM['periodo_ultimo_dato'])
futuro_pd = futuro.select(features).to_pandas()
for c in categoricas:
    futuro_pd[c] = futuro_pd[c].astype("category")

pred_log = modelo_final.predict(futuro_pd, num_iteration=modelo_final.best_iteration)
pred_tn = np.expm1(pred_log)
pred_tn = np.clip(pred_tn, 0, None)

resultado = futuro.select(["product_id"]).to_pandas()
resultado["tn"] = pred_tn

In [10]:
apredecir = pl.read_csv(PARAM['apredecir_path'], separator="\t").to_pandas()

submit = apredecir[["product_id"]].merge(resultado, on="product_id", how="left")
print("nulos en submit (deberian ser 0):", submit["tn"].isna().sum())
submit["tn"] = submit["tn"].fillna(0.0)

archivo_submit = os.path.join(ruta, f"{PARAM['experimento']}_submit.csv")
submit.to_csv(archivo_submit, index=False)
print(archivo_submit)
submit.head()

nulos en submit (deberian ser 0): 0
/home/ds/exp/LGB07_WF/LGB07_WF_submit.csv


,product_id,tn
0,20001,1148.926358
1,20002,1074.597753
2,20003,821.223691
3,20004,599.423271
4,20005,568.384028


In [11]:
def kaggle_submit(competencia, archivo, mensaje):
    comando = f'kaggle competitions submit -c {competencia} -f {archivo} -m "{mensaje}"'
    os.system(comando)

kaggle_submit(PARAM['kaggle_competition'], archivo_submit, f"{PARAM['experimento']} walk-forward")

100%|██████████| 18.7k/18.7k [00:00<00:00, 57.2kB/s]


94 submissions remaining today.
Successfully submitted to Labo III, 2026 BA

In [12]:
importancia = pl.DataFrame({
    "feature": modelo_final.feature_name(),
    "importancia": modelo_final.feature_importance(importance_type="gain")
}).sort("importancia", descending=True)

importancia.write_csv(os.path.join(ruta, "feature_importance.csv"))
print(os.path.join(ruta, "feature_importance.csv"))

/home/ds/exp/LGB07_WF/feature_importance.csv
